## Prompt Templates 

In [4]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, ChatPromptTemplate, PromptTemplate

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
  model='gpt-4o',
  temperature = 0.2,
  )

### Ways of Creating Templates
1. `PromptTemplate`
2. `from_template`
3. `from_messages` 
4. `FewShotPromptTemplate` 

#### `PromptTemplate` class with `input_variables` and `template` parameters

In [5]:
# Using PromptTemplate with `input_variables` and `template` arguments
explanation_prompt = PromptTemplate(
  input_variables = ['concept'],
  template = 'Explain this concept like I\'m 5: {concept}'
)
print(explanation_prompt)

input_variables=['concept'] input_types={} partial_variables={} template="Explain this concept like I'm 5: {concept}"


#### `from_template` method

In [6]:
# ChatPromptTemplate.from_template
explanation_prompt1 = ChatPromptTemplate.from_template(
  'Explain this concept like I\'m 5: {concept}'
)

print(explanation_prompt1)

# PromptTemplate.from_template - this is equivalent to PromptTemplate with `input_variables` and `template` arguments
explanation_prompt2 = PromptTemplate.from_template(
  'Explain this concept like I\'m 5: {concept}'
)
print(explanation_prompt2)

input_variables=['concept'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['concept'], input_types={}, partial_variables={}, template="Explain this concept like I'm 5: {concept}"), additional_kwargs={})]
input_variables=['concept'] input_types={} partial_variables={} template="Explain this concept like I'm 5: {concept}"


In [7]:
explanation_prompt2 == explanation_prompt

True

In [8]:
explanation_prompt3 = ChatPromptTemplate.from_template(
  'Give me {fact_count} facts about the topic: {topic}.'
)

explanation_prompt3_invoked = explanation_prompt3.invoke(
  {
    'fact_count': 10, 
    'topic': 'Dark Matter',
  }
)

response = llm.invoke(
    explanation_prompt3_invoked
)

print(response.content)

Certainly! Here are ten facts about dark matter:

1. **Invisible and Non-Luminous**: Dark matter does not emit, absorb, or reflect any electromagnetic radiation, making it invisible and detectable only through its gravitational effects on visible matter, radiation, and the large-scale structure of the universe.

2. **Massive Component of the Universe**: Dark matter is estimated to make up about 27% of the universe's total mass and energy content. In contrast, ordinary (baryonic) matter, which makes up stars, planets, and living organisms, accounts for only about 5%.

3. **Gravitational Effects**: The presence of dark matter is inferred from its gravitational effects on galaxies and galaxy clusters. It helps explain the rotation curves of galaxies, where the outer regions rotate at unexpected speeds that cannot be accounted for by visible matter alone.

4. **Dark Matter Halos**: Galaxies are thought to be embedded in massive halos of dark matter, which provide the gravitational glue tha

#### `from_messages`

In [9]:
# Using roles in tuples - USE THIS 
# NB: ai = assistant, so we can also use role 'assistant'
messages = [
  ('system', 'You are an expert on the topic: {topic}'),
  ('ai', 'Sure! Can you please specify my knowledge level: {knowledge_level}.'),
  ('human', 'Tell me {fact_count} facts.')
]

tutor_prompt_template = ChatPromptTemplate.from_messages(messages)

# ❌ This does not work! 

# response = llm.invoke(
#   {
#     'topic': 'Dark matter', 
#     'fact_count': '10',
#   }
# )

tutor_prompt_template_invoked = tutor_prompt_template.invoke(
  {'topic': 'dark matter',
   'knowledge_level': 'PhD',
  'fact_count': 8}
)
print(tutor_prompt_template_invoked)
print()

tutor_prompt_template_formatted = tutor_prompt_template.format_messages(
  topic='dark matter',
  knowledge_level='PhD',
  fact_count=8,
)
print(tutor_prompt_template_formatted)
print()
print(tutor_prompt_template)

messages=[SystemMessage(content='You are an expert on the topic: dark matter', additional_kwargs={}, response_metadata={}), AIMessage(content='Sure! Can you please specify my knowledge level: PhD.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me 8 facts.', additional_kwargs={}, response_metadata={})]

[SystemMessage(content='You are an expert on the topic: dark matter', additional_kwargs={}, response_metadata={}), AIMessage(content='Sure! Can you please specify my knowledge level: PhD.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me 8 facts.', additional_kwargs={}, response_metadata={})]

input_variables=['fact_count', 'knowledge_level', 'topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are an expert on the topic: {topic}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input

In [16]:
# Using message classes 
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


messages1 = [
  SystemMessage(content='You are an expert on the topic: {topic}'), 
  AIMessage(content="Sure! I can teach about the topic."),
  HumanMessage(content="Tell me {fact_count} facts.")
]

messages1 = [
  ('system', 'You are an expert on the topic dark matter'), 
  ('ai', 'Sure! I can teach about the topic.'), 
  ('human', 'Tell me 10 facts.')
]

tutor_prompt_template1 = ChatPromptTemplate.from_messages(messages1)
print(tutor_prompt_template1)

# using .invoke
# tutor_prompt_template1_formatted = tutor_prompt_template1.invoke()

# print(tutor_prompt_template1_formatted)

response = llm.invoke(
  messages1
)
print(response)

input_variables=[] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert on the topic dark matter'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Sure! I can teach about the topic.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Tell me 10 facts.'), additional_kwargs={})]
content="Certainly! Here are ten facts about dark matter:\n\n1. **Invisible and Non-Luminous**: Dark matter does not emit, absorb, or reflect light, making it invisible and detectable only through its gravitational effects on visible matter, radiation, and the large-scale structure of the universe.\n\n2. **Abundance**: Dark matter makes up about 27% of the universe's total mass-energy content. In comparison, 

In [11]:
# Mixing tuples and message classes
# This works:
messages2 = [
  ('system', 'You are an expert on the topic: {topic}'), 
  AIMessage(content="Sure! I can help you with that."), 
  ('human', "Tell me {fact_count} facts.")
]

tutor_prompt_template2 = ChatPromptTemplate.from_messages(messages2)
print(tutor_prompt_template2)

tutor_prompt_template2_formatted = tutor_prompt_template2.format_messages(
  topic='dark matter',
  fact_count=10
)
print(tutor_prompt_template2_formatted)

# Invoking 
response2 = llm.invoke(
  tutor_prompt_template2_formatted
)

print(response2.content)

input_variables=['fact_count', 'topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are an expert on the topic: {topic}'), additional_kwargs={}), AIMessage(content='Sure! I can help you with that.', additional_kwargs={}, response_metadata={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['fact_count'], input_types={}, partial_variables={}, template='Tell me {fact_count} facts.'), additional_kwargs={})]
[SystemMessage(content='You are an expert on the topic: dark matter', additional_kwargs={}, response_metadata={}), AIMessage(content='Sure! I can help you with that.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me 10 facts.', additional_kwargs={}, response_metadata={})]
Certainly! Here are ten interesting facts about dark matter:

1. **Invisible and Non-Luminous**: Dark matter does not emit, absorb, or reflect

In [12]:
from langchain.prompts import AIMessagePromptTemplate
messages3 = [
  ('system', 'You are an expert on the topic: {topic}'), 
  # AIMessage(content="Sure! I can help you with that. Can you specify my knowledge level:{knowledge_level}"), 
  AIMessagePromptTemplate.from_template('Sure! I can help you with that. Can you specify my knowledge level: {knowledge_level}'),
  ('human', "Tell me {fact_count} facts.")
]

tutor_prompt_template3 = ChatPromptTemplate.from_messages(messages3)
print(tutor_prompt_template3)

tutor_prompt_template3_formatted = tutor_prompt_template3.format_messages(
  topic='dark matter',
  knowledge_level='PhD', # this did not get inserted into the prompt template when we used just AIMessage() in the messages3
  # we needed to use AIMessagePromptTemplate 
  fact_count=10
)
print(tutor_prompt_template3_formatted)

# Invoking 
response3 = llm.invoke(
  tutor_prompt_template3_formatted
)

print(response2.content)

input_variables=['fact_count', 'knowledge_level', 'topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are an expert on the topic: {topic}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['knowledge_level'], input_types={}, partial_variables={}, template='Sure! I can help you with that. Can you specify my knowledge level: {knowledge_level}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['fact_count'], input_types={}, partial_variables={}, template='Tell me {fact_count} facts.'), additional_kwargs={})]
[SystemMessage(content='You are an expert on the topic: dark matter', additional_kwargs={}, response_metadata={}), AIMessage(content='Sure! I can help you with that. Can you specify my knowledge level: PhD', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tel

In [13]:
messages09 = [
  SystemMessage(content="You are a helpful assistant."),
  HumanMessage(content="What’s the weather like today?"),
]
response = llm.invoke(messages09)
print(response.content)


I'm unable to provide real-time information, including current weather conditions. To find out the weather today, you can check a reliable weather website, use a weather app on your phone, or ask a voice-activated assistant like Siri or Google Assistant.


#### `FewShotPromptTemplate`

In [14]:
from langchain_core.prompts import FewShotPromptTemplate, ChatPromptTemplate

In [15]:
# 1. Define examples in json
examples = [
  {
  'question': 'How many hours does Usama study when he is burnt out?',
  'answer': '2 if burnt out, else 6'
  }, 
  {
    'question': 'What is his favorite place of studying?',
    'answer': 'The main library'
  }, 
  {
    'question': 'What laptop does he use to code?',
    'answer': 'MacBook Pro M1'
  }
]

# 2 Create example prompt template
# example_prompt: PromptTemplate used to format an individual example.
example_prompt = PromptTemplate(
  input_variables=['question', 'answer'], 
  template="Question: {question}\n{answer}"
)

# example_prompt1 = PromptTemplate.from_template("Question: {question}\n{answer}")

# 3 Create few shot prompt 
prompt_template_few_shot = FewShotPromptTemplate(
  examples=examples, 
  example_prompt=example_prompt,
  input_variables=['input'], 
  suffix='Question: {input}'
)
# print(prompt_template_few_shot)
# input_variables: A list of the names of the variables whose values are required as inputs to the prompt.
# suffix: A prompt template string to put after the examples.

# 4. Format template 
prompt_template_few_shot_formatted = prompt_template_few_shot.format_prompt(
  input='How many hours does Usama study and what is his fav place to have a good study sesh?'
)
print(prompt_template_few_shot_formatted)

# 5. Invoke llm
response_few_shot = llm.invoke(prompt_template_few_shot_formatted)
print(response_few_shot.content)


text='Question: How many hours does Usama study when he is burnt out?\n2 if burnt out, else 6\n\nQuestion: What is his favorite place of studying?\nThe main library\n\nQuestion: What laptop does he use to code?\nMacBook Pro M1\n\nQuestion: How many hours does Usama study and what is his fav place to have a good study sesh?'
Usama studies for 6 hours if he is not burnt out, and his favorite place to have a good study session is the main library.


## Memory

External 
1. Vector databases
2. SQL / NoSQL 

Internal 
1. ConversationBufferMemory
2. ConversationTokenBufferMemory
3. ConversationBufferWindowMemory
4. ConversationSummaryMemory

Native
1. List 

### External

#### Firebase: Using Firestore
Firestore is a serverless document-oriented database that scales to meet any demand. Extend your database application to build AI-powered experiences leveraging Firestore's Langchain integrations.

In [17]:
from google.cloud import firestore 
from langchain_google_firestore import FirestoreChatMessageHistory

In [20]:
# FIRESTORE SET UP

# 1. Define relevant variables
PROJECT_ID = 'xai-chatbot-30176'
COLLECTION_NAME = 'chat_history'
SESSION_ID = 'user_session_new'

# 2. Initialising Firestore Client
print('Initialising Firestore Client...')
client = firestore.Client(project=PROJECT_ID)

# 3. Initialise Firestore Chat Messaging History
print("Initialising Firestore Chat Message History...")
chat_history = FirestoreChatMessageHistory(
  session_id = SESSION_ID,
  collection=COLLECTION_NAME,
  client=client,
)

# CHATTING 

# 1. Messages 
messages = [
  SystemMessage(content='You are a pedantic teacher who likes to throw curveballs at his students.'), 
  HumanMessage(content='How can I generate random numbers using Python?'), 
  AIMessage(content='Actually...you cannot. Any idea why?'),
  HumanMessage(content='Uh...I saw online you can do that...')
]

# 2. Loop
while True: 
  query = input('You: ')
  if query.lower() == 'quit': 
    break 
  
  chat_history.add_user_message(query)
  result = llm.invoke(chat_history.messages)

  chat_history.add_ai_message(result)

  print(f"AI: {result.content}")

print(chat_history)

Initialising Firestore Client...
Initialising Firestore Chat Message History...
AI: In Python, you can generate random numbers using the `random` module, which provides various functions for generating random numbers of different types. Here are some common methods:

1. **Random Float Between 0 and 1:**
   - Use `random.random()` to generate a random float number between 0.0 and 1.0.
   ```python
   import random

   random_float = random.random()
   print(random_float)  # Outputs a random float between 0.0 and 1.0
   ```

2. **Random Integer:**
   - Use `random.randint(a, b)` to generate a random integer \( N \) such that \( a \leq N \leq b \).
   ```python
   import random

   random_int = random.randint(1, 10)
   print(random_int)  # Outputs a random integer between 1 and 10, inclusive
   ```

3. **Random Float in a Range:**
   - Use `random.uniform(a, b)` to generate a random float \( N \) such that \( a \leq N \leq b \).
   ```python
   import random

   random_float_range = rando

> NB: You must use `AIMessage`, `HumanMessage`, `SystemMessage` classes directly to store the messages. 

Using `('human', 'message')`  creates a HumanMessagePromptTemplate and cannot be stored as a message

### Internal

In [36]:
from langchain.chains import ConversationChain

#### Conversation Buffer Memory (Deprecated since version 0.3.1)

A basic memory implementation that simply stores the conversation history.

This stores the entire conversation history in memory without any additional processing.

Note that additional processing may be required in some situations when the conversation history is too large to fit in the context window of the model.

In [37]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory()
conversation = ConversationChain(
  llm=llm, 
  memory=memory, 
  verbose=True,
)
conversation.predict(input='Hi many name is Usama.')

/var/folders/pd/13yjfqcd69j8p0x45__ly4700000gn/T/ipykernel_90112/672091806.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/var/folders/pd/13yjfqcd69j8p0x45__ly4700000gn/T/ipykernel_90112/672091806.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi many name is Usama.
AI:

> Finished chain.


"Hello Usama! It's great to meet you. I'm an AI here to chat and help with any questions you might have. How can I assist you today?"

##### Using `.predict` on `ConversationChain`

In [ ]:
conversation.predict(input="What is 1+1?")
# Under the hood, conversation (chain) calls load_memory_variables with {"input": "What is 1+1"}
# and injects the returned {"history": ...} into the prompt.



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi many name is Usama.
AI: Hello Usama! It's great to meet you. I'm an AI here to chat and help with any questions you might have. How can I assist you today?
Human: What is 1+1?
AI:

> Finished chain.


"1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to know, feel free to ask!"

In [39]:
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi many name is Usama.
AI: Hello Usama! It's great to meet you. I'm an AI here to chat and help with any questions you might have. How can I assist you today?
Human: What is 1+1?
AI: 1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to know, feel free to ask!
Human: What is my name?
AI:

> Finished chain.


"Your name is Usama! You mentioned it at the beginning of our conversation. If there's anything else you'd like to discuss or ask about, I'm here to help!"

Printing the memory buffer: 
`memory.buffer`

NB: We get the `buffer` attribute from the `memory` object 

In [40]:
print(memory.buffer)

Human: Hi many name is Usama.
AI: Hello Usama! It's great to meet you. I'm an AI here to chat and help with any questions you might have. How can I assist you today?
Human: What is 1+1?
AI: 1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to know, feel free to ask!
Human: What is my name?
AI: Your name is Usama! You mentioned it at the beginning of our conversation. If there's anything else you'd like to discuss or ask about, I'm here to help!


##### Loading memory variables using `load_memory_variables({})`

In [46]:
memory.load_memory_variables({})

{'history': "Human: Hi many name is Usama.\nAI: Hello Usama! It's great to meet you. I'm an AI here to chat and help with any questions you might have. How can I assist you today?\nHuman: What is 1+1?\nAI: 1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to know, feel free to ask!\nHuman: What is my name?\nAI: Your name is Usama! You mentioned it at the beginning of our conversation. If there's anything else you'd like to discuss or ask about, I'm here to help!"}

##### Using `.save_context()` method on `memory` instance

In [ ]:
memory = ConversationBufferMemory(
  memory_key='hist', 
  return_messages=True,
)
memory.save_context(
  {"Wagwan": "Wagwan"}, 
  {"Yeehaw!!": "You good?"}, 
  )
print('Printing Memory Buffer....')
print(memory.buffer)
print()
print('Printing memory variables....')
memory.load_memory_variables({})

Printing Memory Buffer....
[HumanMessage(content='Wagwan', additional_kwargs={}, response_metadata={}), AIMessage(content='You good?', additional_kwargs={}, response_metadata={})]

Printing memory variables....


{'hist': [HumanMessage(content='Wagwan', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You good?', additional_kwargs={}, response_metadata={})]}

In [ ]:
# Adding more context 
memory.save_context(
  {
    'input': 'Not much. Just Joshin.'
  }, 
  {
    'robot': 'Ayyy'
  }
)
memory.load_memory_variables({})

{'hist': [HumanMessage(content='Wagwan', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You good?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Not much. Just {joshin}.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ayyy', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Not much. Just {joshin}.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ayyy', additional_kwargs={}, response_metadata={})]}

#### Conversation Token Buffer Memory (Deprecated since version 0.3.1)

Conversation chat memory with token limit.

Keeps only the most recent messages in the conversation under the constraint that the total number of tokens in the conversation does not exceed a certain limit.

NB: The `llm` parameter is required since different llms define tokens differently

In [79]:
from langchain.memory import ConversationTokenBufferMemory
memory1 = ConversationTokenBufferMemory(llm=llm, max_token_limit=50, return_messages=False)
memory1.save_context({"input": "AI is what?!"},
  {"output": "Amazing!"})
memory1.save_context({"input": "Backpropagation is what?"},
  {"output": "Beautiful!"})
memory1.save_context({"input": "Chatbots are what?"}, 
  {"output": "Charming!"})

vars = memory1.load_memory_variables({})
print(vars.get('history'))

Human: AI is what?!
AI: Amazing!
Human: Backpropagation is what?
AI: Beautiful!
Human: Chatbots are what?
AI: Charming!


#### Conversation Buffer Window Memory (Deprecated since version 0.3.1)

Used to keep track of the last k turns of a conversation

If the number of messages is more than the maximum number of messages to keep, the oldest messages are dropped

In [101]:
from langchain.memory import ConversationBufferWindowMemory
memory2 = ConversationBufferWindowMemory(
  memory_key='history',
  return_messages=False,
  k=2,
)
memory2.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory2.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory2.load_memory_variables({})

conversation2 = ConversationChain(
  llm=llm, 
  memory=memory2,
  verbose=True
)

conversation2.predict(input="Hi, my name is Usama")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi
AI: What's up
Human: Not much, just hanging
AI: Cool
Human: Hi, my name is Usama
AI:

> Finished chain.


"Hi Usama! It's great to meet you. I'm an AI, here to chat and help with any questions you might have. What are you up to today?"

In [102]:
conversation2.predict(input="What is 1+1?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Not much, just hanging
AI: Cool
Human: Hi, my name is Usama
AI: Hi Usama! It's great to meet you. I'm an AI, here to chat and help with any questions you might have. What are you up to today?
Human: What is 1+1?
AI:

> Finished chain.


"1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to discuss, feel free to ask!"

In [103]:
conversation2.predict(input="What are quarks?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Usama
AI: Hi Usama! It's great to meet you. I'm an AI, here to chat and help with any questions you might have. What are you up to today?
Human: What is 1+1?
AI: 1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to discuss, feel free to ask!
Human: What are quarks?
AI:

> Finished chain.


'Quarks are fundamental particles that are the building blocks of matter. They combine to form protons and neutrons, which in turn make up the nuclei of atoms. Quarks are part of the Standard Model of particle physics, which is the theory describing the fundamental forces and particles in the universe.\n\nThere are six types, or "flavors," of quarks: up, down, charm, strange, top, and bottom. Each of these has a unique mass and charge. For example, protons are made of two up quarks and one down quark, while neutrons consist of two down quarks and one up quark.\n\nQuarks are never found in isolation; they are always bound together by the strong force, mediated by particles called gluons. This phenomenon is known as "confinement." If you have more questions about quarks or particle physics, feel free to ask!'

In [104]:
conversation2.predict(input="What about neutrinos?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: What is 1+1?
AI: 1+1 equals 2. It's one of the most basic arithmetic operations and a fundamental building block in mathematics. If you have any more math questions or anything else you'd like to discuss, feel free to ask!
Human: What are quarks?
AI: Quarks are fundamental particles that are the building blocks of matter. They combine to form protons and neutrons, which in turn make up the nuclei of atoms. Quarks are part of the Standard Model of particle physics, which is the theory describing the fundamental forces and particles in the universe.

There are six types, or "flavors," of quarks: up, down, charm, strange, top, and bottom. Each of these has a unique

'Neutrinos are fascinating subatomic particles that are part of the lepton family in the Standard Model of particle physics. They are incredibly light, neutral particles that interact only via the weak nuclear force and gravity, making them extremely difficult to detect. There are three known types, or "flavors," of neutrinos: electron neutrinos, muon neutrinos, and tau neutrinos, each associated with their corresponding charged leptons (electron, muon, and tau).\n\nOne of the most intriguing aspects of neutrinos is their ability to oscillate between these flavors as they travel through space. This phenomenon, known as neutrino oscillation, implies that neutrinos have a small but nonzero mass, which was a surprising discovery because it required an extension of the original Standard Model.\n\nNeutrinos are produced in a variety of processes, such as nuclear reactions in the sun, during supernovae, and in nuclear reactors. Despite their abundance in the universe, their weak interactions

In [105]:
conversation2.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: What are quarks?
AI: Quarks are fundamental particles that are the building blocks of matter. They combine to form protons and neutrons, which in turn make up the nuclei of atoms. Quarks are part of the Standard Model of particle physics, which is the theory describing the fundamental forces and particles in the universe.

There are six types, or "flavors," of quarks: up, down, charm, strange, top, and bottom. Each of these has a unique mass and charge. For example, protons are made of two up quarks and one down quark, while neutrons consist of two down quarks and one up quark.

Quarks are never found in isolation; they are always bound together by the strong fo

"I'm sorry, but I don't have access to personal information, so I don't know your name. If you'd like, you can tell me your name or any other details you'd like to share!"

In [106]:
print(memory2.buffer)

Human: What about neutrinos?
AI: Neutrinos are fascinating subatomic particles that are part of the lepton family in the Standard Model of particle physics. They are incredibly light, neutral particles that interact only via the weak nuclear force and gravity, making them extremely difficult to detect. There are three known types, or "flavors," of neutrinos: electron neutrinos, muon neutrinos, and tau neutrinos, each associated with their corresponding charged leptons (electron, muon, and tau).

One of the most intriguing aspects of neutrinos is their ability to oscillate between these flavors as they travel through space. This phenomenon, known as neutrino oscillation, implies that neutrinos have a small but nonzero mass, which was a surprising discovery because it required an extension of the original Standard Model.

Neutrinos are produced in a variety of processes, such as nuclear reactions in the sun, during supernovae, and in nuclear reactors. Despite their abundance in the unive

#### Conversation Summary Memory (Deprecated since version 0.3.1)

Continually summarises the conversation history. 

Summary is updated after each conversation turn.

NB: that max token budget applies to the combined content of:

The running summary (“System: …”)

The most recent raw messages still in the buffer

In [111]:
from langchain.memory import ConversationSummaryBufferMemory
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

memory3 = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory3.save_context({"input": "Hello"}, {"output": "What's up"})
memory3.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory3.save_context({"input": "What is on the schedule today?"}, 
                    {"output": f"{schedule}"})

In [113]:
conversation3 = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)
conversation3.predict(input="What would be a good demo to show?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human greets the AI, and they exchange casual pleasantries. The human then asks what is on the schedule for today.
AI: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.
Human: What would be a good demo to show?
AI:

> Finished chain.


"For your lunch meeting, a great demo to show would be a real-time language translation using a large language model (LLM). This can effectively showcase the power and versatility of AI in breaking down language barriers. You could demonstrate how the model can translate a conversation between two different languages almost instantaneously, highlighting its potential applications in global communication and business.\n\nAnother impressive demo could be using the LLM to generate creative content, such as writing a short story or poem based on a few prompts. This would illustrate the AI's ability to understand context and generate coherent and creative text, which can be particularly engaging for someone interested in the latest AI advancements.\n\nIf your customer is interested in more technical aspects, you could demonstrate how the LLM can assist in coding by generating code snippets or debugging existing code. This would showcase its utility in software development and how it can enh

In [116]:
vars = memory3.load_memory_variables({})
print(len(vars['history'].split(' ')))

107


## Output Parsing 

In [ ]:
# The format we want 
{
  "player": 'Leo Messi',
  "goals": 10,
  "passes": 5,
  "highlights": "great player!"
}

match_summary = "Inter Miami CF 3–2 LAFC\nDate: July 12, 2025\nVenue: DRV PNK Stadium, Fort Lauderdale\n\nFirst Half\n\n- Kick-off & Early Momentum (0’–15’): LAFC, playing with a high press, carved out two half-chances in the opening ten minutes. At 11’, Diego Rossi burst in behind the back line but his low drive was smothered by Drake Callender.\n- Opening Goal (18’): Inter Miami’s breakthrough came when Jordi Alba swung in a pinpoint corner from the left. Nicolás Freire rose highest to nod home at the near post—1–0.\n- LAFC Response (26’): LAFC equalized on the counter: a quick one-two between Cristian Arango and Kwadwo Opoku, with Opoku slotting past Callender—1–1.\n- Messi’s First Touch of Note (32’): Picking the ball just inside the LAFC half, Lionel Messi threaded a clever through-ball to Luis Suárez, whose pull-back was just off the pace. A sign of things to come.\n\nSecond Half\n\n- Miami Retakes the Lead (52’): Messi opened his account in the 52nd minute. After combining with Alba on the left flank, he danced past a defender and curled a perfect right-footed strike into the top corner—2–1.\n- Drama in the Middle (60’–70’): The game opened up as both sides committed bodies forward. LAFC’s Latif Blessing forced a smart save from Callender at 63’, and Miami’s Suárez rattled a post three minutes later.\n- Messi’s Assist (71’): Messi collected the ball at the edge of the box, shrugged off a tackle, then delivered a low pass into the path of Federico Bernardeschi, who tapped in for 3–1.\n- Late LAFC Rally (81’ & 89’): LAFC weren’t done: at 81’, a thunderous header by Sean Goldberg made it 3–2. Then in the 89th minute, a brilliant solo run by Arango ended with a close-range finish, but VAR ruled it offside by inches.\n\nKey Statistics\n\nTeam               | Inter Miami CF | LAFC      \n--------------------|---------------:|----------:\nPossession         |      56%       |     44%   \nTotal Shots        |      18        |     12    \nShots on Target    |      9         |     5     \nCorners            |      7         |     3     \nFouls Committed    |      14        |     11    \n\nMessi’s Match Contributions\n\n- Goals: 1 (52’)\n- Assists: 1 (71’)\n- Key Passes: 4\n- Dribbles Completed: 6/8\n- Pass Accuracy: 90% (62/69)\n\nMain Highlights\n\n1. Set-Piece Success: Freire’s opener from Alba’s corner set the tone for Miami’s aerial strength.\n2. Swift Counters: LAFC’s equalizer in the first half epitomized their lightning-quick transition play.\n3. Messi Magic: His goal was a thing of beauty—quick feet, clinical finish—and his vision unlocked the defense for the third.\n4. End-to-End Thrill: Both keepers were kept busy; two disallowed goals and a post rattled underscored how fine the margins were.\n5. Decisive VAR Decisions: Late in the game, VAR intervened to chalk off an 89th-minute goal, preserving Miami’s slender advantage.\n\nThis 3–2 win cements Inter Miami’s push for the Supporters’ Shield, while LAFC will rue missed opportunities to snatch a point on the road."

In [ ]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
# 1. Define schemas 
player_schema = ResponseSchema(
  name='player', 
  description='The name of the player in the match summary. If there are multiple, select the one most mentioned.',
)

goals_schema = ResponseSchema(
  name='goals', 
  description='The number of goals scored by the player with most mentioned name.', 
  type='int'
)

assists_schema = ResponseSchema(
  name='assists', 
  description='The number of assists made by the player with most mentioned name.', 
  type='int'
)

highlights_schema = ResponseSchema(
  name='player_highlights', 
  description="Extract any sentences about the player in the summary. Make a dictionary of the highlights where the key is the type of highlight, e.g., 'foul' and the value is the description", 
  type='dict'
)

response_schemas = [player_schema, goals_schema, assists_schema, highlights_schema]
print(response_schemas[-1])

name='player_highlights' description="Extract any sentences about the player in the summary. Make a dictionary of the highlights where the key is the type of highlight, e.g., 'foul' and the value is the description" type='dict'


In [125]:
# 2. Instantiate output parser
output_parser = StructuredOutputParser.from_response_schemas(response_schemas=response_schemas)

# 3. Retrieve format instructions to put into the prompt
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"player": string  // The name of the player in the match summary. If there are multiple, select the one most mentioned.
	"goals": int  // The number of goals scored by the player with most mentioned name.
	"assists": int  // The number of assists made by the player with most mentioned name.
	"player_highlights": dict  // Extract any sentences about the player in the summary. Make a dictionary of the highlights where the key is the type of highlight, e.g., 'foul' and the value is the description
}
```


In [126]:
# 4. Define prompt template with format instructions 
player_summary_template = """\
For the following text, extract the following information:

player: Who is the most mentioned player?

goals: How many goals did the most mentioned player score?

assists: How many assists did the most mentioned player make?

text: {text}

{format_instructions}
"""

player_summary_prompt = PromptTemplate(
  input_variables=['text', 'format_instructions'],
  template=player_summary_template
)

player_summary_prompt_formatted = player_summary_prompt.format_prompt(
  text=match_summary,
  format_instructions=format_instructions
)
print(player_summary_prompt_formatted)

text='For the following text, extract the following information:\n\nplayer: Who is the most mentioned player?\n\ngoals: How many goals did the most mentioned player score?\n\nassists: How many assists did the most mentioned player make?\n\ntext: Inter Miami CF 3–2 LAFC\nDate: July 12, 2025\nVenue: DRV PNK Stadium, Fort Lauderdale\n\nFirst Half\n\n- Kick-off & Early Momentum (0’–15’): LAFC, playing with a high press, carved out two half-chances in the opening ten minutes. At 11’, Diego Rossi burst in behind the back line but his low drive was smothered by Drake Callender.\n- Opening Goal (18’): Inter Miami’s breakthrough came when Jordi Alba swung in a pinpoint corner from the left. Nicolás Freire rose highest to nod home at the near post—1–0.\n- LAFC Response (26’): LAFC equalized on the counter: a quick one-two between Cristian Arango and Kwadwo Opoku, with Opoku slotting past Callender—1–1.\n- Messi’s First Touch of Note (32’): Picking the ball just inside the LAFC half, Lionel Messi

In [133]:
# 5. Invoke LLM
response4 = llm.invoke(player_summary_prompt_formatted) 

print(response4.content)

```json
{
	"player": "Lionel Messi",
	"goals": 1,
	"assists": 1,
	"player_highlights": {
		"first_touch_of_note": "Picking the ball just inside the LAFC half, Lionel Messi threaded a clever through-ball to Luis Suárez, whose pull-back was just off the pace. A sign of things to come.",
		"goal": "Messi opened his account in the 52nd minute. After combining with Alba on the left flank, he danced past a defender and curled a perfect right-footed strike into the top corner—2–1.",
		"assist": "Messi collected the ball at the edge of the box, shrugged off a tackle, then delivered a low pass into the path of Federico Bernardeschi, who tapped in for 3–1.",
		"match_contributions": "Goals: 1 (52’), Assists: 1 (71’), Key Passes: 4, Dribbles Completed: 6/8, Pass Accuracy: 90% (62/69)",
		"magic": "His goal was a thing of beauty—quick feet, clinical finish—and his vision unlocked the defense for the third."
	}
}
```


In [134]:
# Parse the output
output_dict = output_parser.parse(response4.content)
output_dict

{'player': 'Lionel Messi',
 'goals': 1,
 'assists': 1,
 'player_highlights': {'first_touch_of_note': 'Picking the ball just inside the LAFC half, Lionel Messi threaded a clever through-ball to Luis Suárez, whose pull-back was just off the pace. A sign of things to come.',
  'goal': 'Messi opened his account in the 52nd minute. After combining with Alba on the left flank, he danced past a defender and curled a perfect right-footed strike into the top corner—2–1.',
  'assist': 'Messi collected the ball at the edge of the box, shrugged off a tackle, then delivered a low pass into the path of Federico Bernardeschi, who tapped in for 3–1.',
  'match_contributions': 'Goals: 1 (52’), Assists: 1 (71’), Key Passes: 4, Dribbles Completed: 6/8, Pass Accuracy: 90% (62/69)',
  'magic': 'His goal was a thing of beauty—quick feet, clinical finish—and his vision unlocked the defense for the third.'}}

In [139]:
type(output_dict.get('goals'))

int

## Chains

NB: When we are LCEL for chain expressions, we don't need to format the prompts.

The LangChain Expression Language (LCEL) is a declarative way to compose Runnables into chains. Any chain constructed this way will automatically have sync, async, batch, and streaming support.

The main composition primitives are `RunnableSequence` and `RunnableParallel`.

RunnableSequence invokes a series of runnables sequentially, with one Runnable’s output serving as the next’s input. Construct using the | operator or by passing a list of runnables to RunnableSequence.

RunnableParallel invokes runnables concurrently, providing the same input to each. Construct it using a dict literal within a sequence or by passing a dict to RunnableParallel.

### Sequential Chains

#### Simple Sequential Chains

##### Using LCEL

In [146]:
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv()

code_llm = ChatOllama(
  model='qwen2.5-coder:7b', 
  base_url='http://localhost:11434'
)

code_prompt = ChatPromptTemplate.from_messages(
  [
    ('system', 'You are an expert coder and a pedantic tutor who teaches every concept involved in a programming problem / issue'),
  ('human', 'Solve this problem for me {problem}'),
  ]
)

decomposition_prompt = ChatPromptTemplate.from_template(
  "Break the solution / explanation into steps: {solution}"
)

questions_prompt = ChatPromptTemplate.from_template(
  "Generate a Socratic question for each of the steps: {steps}"
)

seq_chain = (
  {'solution': code_prompt | code_llm | StrOutputParser()}
  | decomposition_prompt 
  | llm 
  | StrOutputParser()
  | questions_prompt
  | llm 
  | StrOutputParser()
)

response_code_questions = seq_chain.invoke(
  {
    'problem': """
      Given an array of strings words and a width maxWidth, format the text such that each line has exactly maxWidth characters and is fully (left and right) justified.

You should pack your words in a greedy approach; that is, pack as many words as you can in each line. Pad extra spaces ' ' when necessary so that each line has exactly maxWidth characters.

Extra spaces between words should be distributed as evenly as possible. If the number of spaces on a line does not divide evenly between words, the empty slots on the left will be assigned more spaces than the slots on the right.

For the last line of text, it should be left-justified, and no extra space is inserted between words.

Note:

A word is defined as a character sequence consisting of non-space characters only.
Each word's length is guaranteed to be greater than 0 and not exceed maxWidth.
The input array words contains at least one word.
 

Example 1:

Input: words = ["This", "is", "an", "example", "of", "text", "justification."], maxWidth = 16
Output:
[
   "This    is    an",
   "example  of text",
   "justification.  "
]
Example 2:

Input: words = ["What","must","be","acknowledgment","shall","be"], maxWidth = 16
Output:
[
  "What   must   be",
  "acknowledgment  ",
  "shall be        "
]
Explanation: Note that the last line is "shall be    " instead of "shall     be", because the last line must be left-justified instead of fully-justified.
Note that the second line is also left-justified because it contains only one word.
Example 3:

Input: words = ["Science","is","what","we","understand","well","enough","to","explain","to","a","computer.","Art","is","everything","else","we","do"], maxWidth = 20
Output:
[
  "Science  is  what we",
  "understand      well",
  "enough to explain to",
  "a  computer.  Art is",
  "everything  else  we",
  "do                  "
]

Constraints:

1 <= words.length <= 300
1 <= words[i].length <= 20
words[i] consists of only English letters and symbols.
1 <= maxWidth <= 100
words[i].length <= maxWidth
    """
  }
)
print(response_code_questions) 

Certainly! Here are Socratic questions for each step of the text justification problem:

### Problem Breakdown

1. **Objective**:
   - What does it mean for a line of text to be "fully justified," and how does this differ from "left-justified"?

2. **Approach**:
   - How can a greedy algorithm help in determining how many words fit into a line without exceeding the maximum width?
   - Why is it important to handle the last line differently from the other lines when justifying text?

### Step-by-Step Solution

1. **Initialize Variables**:
   - What role do the `result`, `currentLine`, and `currentWidth` variables play in the process of text justification?

2. **Iterate Through Words**:
   - How do you determine whether adding a new word to the current line will exceed the `maxWidth`?
   - What should be done when a word cannot be added to the current line without exceeding the `maxWidth`?

3. **Finalize Current Line**:
   - How do you calculate the number of spaces needed to ensure the 

##### Using `RunnableSequence`

In [ ]:
from langchain_core.runnables.base import RunnableLambda, RunnableSequence

# ---- 0. Define prompts ----
code_prompt = ChatPromptTemplate.from_messages(
  [
    ('system', 'You are an expert coder and a pedantic tutor who teaches every concept involved in a programming problem / issue'),
  ('human', 'Solve this problem for me {problem}'),
  ]
)
decomposition_prompt = ChatPromptTemplate.from_template(
  "Break the solution / explanation into questions: {solution}" \
  "But don't answer them"
)

# ---- 1. Define Runnable Lambdas (tasks) for Code Generation ----
format_code_prompt = RunnableLambda(lambda x:code_prompt.format_prompt(**x))
code_model_invocation = RunnableLambda(lambda x:code_llm.invoke(x))
output_parse = RunnableLambda(lambda x:x.content)

# ---- 2. Define Runnable Lambdas for Steps Generation ----
format_decomposition_prompt = RunnableLambda(lambda x:decomposition_prompt.format_prompt(**x)) 
decomposition_model_invocation = RunnableLambda(lambda x:llm.invoke(x))

# ---- 3. Create RunnableSequences (i.e., chain for code and decomposition steps) ----
code_chain = RunnableSequence(
  first=format_code_prompt,
  middle=[code_model_invocation],
  last=output_parse,
)

decomposition_chain = RunnableSequence(
  first=format_decomposition_prompt,
  middle=[decomposition_model_invocation],
  last=output_parse,
)

# ---- 4. Chain RunnableSequences Using | ----
final_chain = {'solution': code_chain} | decomposition_chain  

response_code_chain = final_chain.invoke(
  {'problem': 'Recursive merge sort'}
)
print(response_code_chain)

Certainly! Here are the questions based on the explanation provided:

1. **What is the main concept behind the merge sort algorithm?**

2. **What are the three main steps involved in the merge sort process?**

3. **What is the base case for the recursive `merge_sort` function?**

4. **How do you determine the middle point to divide the array into two halves?**

5. **How does the `merge_sort` function recursively sort each half of the array?**

6. **What is the purpose of the `merge` function in the merge sort algorithm?**

7. **How are elements compared and merged in the `merge` function?**

8. **What happens to the remaining elements in the left or right half after the main comparison loop in the `merge` function?**

9. **What is the time complexity of the merge sort algorithm and why?**

10. **What is the space complexity of the merge sort algorithm and why?**

11. **Can you explain the overall process of the merge sort algorithm using an example array?**

12. **How does the merge so

#### Sequential Chains

In [209]:
from langchain.chains.sequential import SequentialChain
from langchain_core.runnables.base import RunnableParallel, RunnableLambda, RunnableSequence

# chain_one = first_prompt | llm | StrOutputParser()

review="""
Produkt: NoiseCanceller Pro X200 Bluetooth-Kopfhörer
Bewertung: ★★★★☆ (4/5)

Titel: Hervorragender Tragekomfort mit starker Geräuschunterdrückung

Positiv:

Exzellente Geräuschunterdrückung: Umgebungsgeräusche werden effektiv herausgefiltert, ideal für Büro, Bahn oder Flugzeug.

Komfortables Design: Weiche Memory-Foam-Ohrenpolster und verstellbarer Kopfbügel sorgen auch bei längerem Tragen für Bequemlichkeit.

Guter Klang: Ausgewogene Klangsignatur mit satten Bässen und klaren Höhen macht Musikgenuss zum Vergnügen.

Lange Akkulaufzeit: Bis zu 30 Stunden Wiedergabe (ANC aus) bzw. 20 Stunden mit aktivierter Geräuschunterdrückung.

Negativ:

Mikrofonqualität: Stimmenübertragung ist etwas dumpf; für wichtige Telefonate nicht optimal.

Gewicht: Mit 280 g fallen sie etwas schwerer aus als manch andere Modelle in dieser Preisklasse.

App-Funktionalität: Die Begleit-App ist etwas eingeschränkt und bietet nur wenige Equalizer-Voreinstellungen.

Fazit:
Die NoiseCanceller Pro X200 bieten ein erstklassiges Hörerlebnis mit sehr guter Geräuschunterdrückung und hohem Tragekomfort. Wer auf perfekte Sprachübertragung und supersleichtes Design verzichten kann, erhält hier einen zuverlässigen Begleiter für Alltag und Reisen. Mit einem Preis von circa 180 € ist das Preis-Leistungs-Verhältnis überzeugend."""

# def get_translation (review): 
#   translation_prompt = PromptTemplate(
#   input_variables=['review'],
#   template=""" 
#     Translate the following review to English: {review}""",
# )
#   return translation_prompt.format_prompt(**review)

# translation_chain = (
#   RunnableLambda(lambda x:get_translation(x)) 
# | llm 
# | StrOutputParser()
# )

# def get_summary(english_review):
#     summary_prompt = ChatPromptTemplate.from_template(
#       """ 
#         Can you summarize the following review in one sentence: {english_review}
#       """
#     )
#     return summary_prompt.format_prompt(english_review=english_review)

# summary_chain = (
#   RunnableLambda(lambda x:get_summary(x)) 
# | llm 
# | StrOutputParser()
# )

translation_prompt = PromptTemplate(
  input_variables=['review'],
  template=""" 
    Translate the following review to English: {review}""",
)

translation_chain = translation_prompt | llm | StrOutputParser()

summary_prompt = ChatPromptTemplate.from_template(
      """ 
        Can you summarize the following review in one sentence: {english_review}
      """
)

summary_chain = summary_prompt | llm | StrOutputParser()
intermediate_chain = translation_chain | summary_chain


# def get_language(review): 
#   language_prompt = ChatPromptTemplate.from_template(
#      'What language is this review in {review}'
#   )
#   return language_prompt.format_prompt(**review)

language_prompt = ChatPromptTemplate.from_template(
     'What language is this review in {review}'
  )

langugage_chain = (language_prompt
| llm 
| StrOutputParser())


def summary_and_language(language, summary): 
   return {
      'summary': summary, 
      'language': language
   }

prompt_four = ChatPromptTemplate.from_template(
  "Write a follow up response to the following summary in the specified language: {summary} {language} "
)
chain_four = prompt_four | llm | StrOutputParser()

overall_seq_chain = (RunnableParallel(branches={'summary': intermediate_chain, 'language': langugage_chain})
| RunnableLambda(lambda x: summary_and_language(x['branches']['language'], x['branches']['summary']))
| chain_four
)
repsonse_seq_chain = overall_seq_chain.invoke({'review': review})

print(repsonse_seq_chain)

Vielen Dank für Ihre detaillierte Zusammenfassung der NoiseCanceller Pro X200 Bluetooth-Kopfhörer. Es freut mich zu hören, dass die Kopfhörer in den Bereichen Geräuschunterdrückung, Komfort und Klangqualität überzeugen können. Der etwas schwerere Aufbau und die eingeschränkte App-Funktionalität sind sicherlich Aspekte, die man vor dem Kauf berücksichtigen sollte. Auch die Mikrofonqualität scheint verbesserungswürdig zu sein, was für Nutzer, die häufig Anrufe tätigen, relevant sein könnte. Dennoch scheint der Preis von 180 € angesichts der gebotenen Leistung fair zu sein, insbesondere für Musikliebhaber und Reisende. Haben Sie die Kopfhörer selbst ausprobiert, oder basiert Ihre Einschätzung auf anderen Quellen?


##### Using `RunnablePassthrough.assign`

In [186]:
from langchain.chains.sequential import SequentialChain
from langchain_core.runnables.base import RunnableParallel, RunnableLambda, RunnableSequence
from langchain_core.runnables import RunnablePassthrough

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}")
chain_one = first_prompt | llm | StrOutputParser()

second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
chain_two = second_prompt | llm | StrOutputParser()

third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = third_prompt | llm | StrOutputParser()

fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {Summary}\n\nLanguage: {Language}"
)
chain_four = fourth_prompt | llm | StrOutputParser()

final_seq_chain = RunnablePassthrough.assign(
  English_Review=chain_one,
  Language=chain_three
).assign(
  Summary=chain_two
).assign(
  follow_up_response=chain_four
)

final_seq_chain_response = final_seq_chain.invoke({'Review':review})
print(final_seq_chain_response)

{'Review': '\nProdukt: NoiseCanceller Pro X200 Bluetooth-Kopfhörer\nBewertung: ★★★★☆ (4/5)\n\nTitel: Hervorragender Tragekomfort mit starker Geräuschunterdrückung\n\nPositiv:\n\nExzellente Geräuschunterdrückung: Umgebungsgeräusche werden effektiv herausgefiltert, ideal für Büro, Bahn oder Flugzeug.\n\nKomfortables Design: Weiche Memory-Foam-Ohrenpolster und verstellbarer Kopfbügel sorgen auch bei längerem Tragen für Bequemlichkeit.\n\nGuter Klang: Ausgewogene Klangsignatur mit satten Bässen und klaren Höhen macht Musikgenuss zum Vergnügen.\n\nLange Akkulaufzeit: Bis zu 30 Stunden Wiedergabe (ANC aus) bzw. 20 Stunden mit aktivierter Geräuschunterdrückung.\n\nNegativ:\n\nMikrofonqualität: Stimmenübertragung ist etwas dumpf; für wichtige Telefonate nicht optimal.\n\nGewicht: Mit 280 g fallen sie etwas schwerer aus als manch andere Modelle in dieser Preisklasse.\n\nApp-Funktionalität: Die Begleit-App ist etwas eingeschränkt und bietet nur wenige Equalizer-Voreinstellungen.\n\nFazit:\nDie N

### Routing

### Branching

In [168]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate


# Step 1: Translate to English
chain_one = (
    ChatPromptTemplate.from_template("Translate the following review to English:\n\n{review}")
    | llm
    | StrOutputParser()
)

# Step 2: Summarize the English review
chain_two = (
    ChatPromptTemplate.from_template("Can you summarize the following review in one sentence:\n\n{english_review}")
    | llm
    | StrOutputParser()
)

# Step 3: Detect original language
chain_three = (
    ChatPromptTemplate.from_template("What language is the following review in:\n\n{review}")
    | llm
    | StrOutputParser()
)

# Step 4: Generate a follow-up message in that language
chain_four = (
    PromptTemplate.from_template(
        "Write a follow up response to the following summary in the specified language:\n\nSummary: {summary}\nLanguage: {language}"
    )
    | llm
    | StrOutputParser()
)

# Chain them together with explicit output key mapping
pipeline = (
    first_prompt | llm | StrOutputParser() 
)

review_text = """
Produkt: NoiseCanceller Pro X200 Bluetooth-Kopfhörer
Bewertung: ★★★★☆ (4/5)

Titel: Hervorragender Tragekomfort mit starker Geräuschunterdrückung

Positiv:
Exzellente Geräuschunterdrückung: Umgebungsgeräusche werden effektiv herausgefiltert, ideal für Büro, Bahn oder Flugzeug.
Komfortables Design: Weiche Memory-Foam-Ohrenpolster und verstellbarer Kopfbügel sorgen auch bei längerem Tragen für Bequemlichkeit.
Guter Klang: Ausgewogene Klangsignatur mit satten Bässen und klaren Höhen macht Musikgenuss zum Vergnügen.
Lange Akkulaufzeit: Bis zu 30 Stunden Wiedergabe (ANC aus) bzw. 20 Stunden mit aktivierter Geräuschunterdrückung.

Negativ:
Mikrofonqualität: Stimmenübertragung ist etwas dumpf; für wichtige Telefonate nicht optimal.
Gewicht: Mit 280 g fallen sie etwas schwerer aus als manch andere Modelle in dieser Preisklasse.
App-Funktionalität: Die Begleit-App ist etwas eingeschränkt und bietet nur wenige Equalizer-Voreinstellungen.

Fazit:
Die NoiseCanceller Pro X200 bieten ein erstklassiges Hörerlebnis mit sehr guter Geräuschunterdrückung und hohem Tragekomfort. Wer auf perfekte Sprachübertragung und supersleichtes Design verzichten kann, erhält hier einen zuverlässigen Begleiter für Alltag und Reisen. Mit einem Preis von circa 180 € ist das Preis-Leistungs-Verhältnis überzeugend.
"""

result = pipeline.invoke({"review": review_text})
print("📝 Final response:\n", result)


📝 Final response:
 Product: NoiseCanceller Pro X200 Bluetooth Headphones  
Rating: ★★★★☆ (4/5)

Title: Excellent Comfort with Strong Noise Cancellation

Positive:  
- Excellent Noise Cancellation: Ambient noises are effectively filtered out, ideal for the office, train, or airplane.  
- Comfortable Design: Soft memory foam ear cushions and an adjustable headband ensure comfort even during extended wear.  
- Good Sound: Balanced sound signature with rich bass and clear highs makes music enjoyment a pleasure.  
- Long Battery Life: Up to 30 hours of playback (ANC off) or 20 hours with active noise cancellation.

Negative:  
- Microphone Quality: Voice transmission is somewhat muffled; not optimal for important calls.  
- Weight: At 280 g, they are somewhat heavier than some other models in this price range.  
- App Functionality: The accompanying app is somewhat limited and offers only a few equalizer presets.

Conclusion:  
The NoiseCanceller Pro X200 offers a first-class listening expe